# Project Setup — the MCP CLI chatbot

The rest of this section builds a **CLI chatbot** that talks to documents through MCP. Unlike earlier sections, the code lives in a **standalone project** (not notebook cells): [`./cli-project/`](./cli-project). This notebook documents how to get it running so a fresh clone knows exactly what to do before the next lesson.

**What it is:** an MCP **client** (handles the chat/user interaction) + a custom MCP **server** (exposes document tools/resources/prompts). Documents are stored in memory — no database. In the real world you'd build *one* side; here we build both to see them communicate.

## Project layout

```
cli-project/
├── main.py            # entrypoint: wires client + server + chat loop
├── mcp_server.py      # the MCP SERVER (tools / resources / prompts) — we fill this in
├── mcp_client.py      # the MCP CLIENT (list/call tools, prompts, resources) — we fill this in
├── core/
│   ├── claude.py      # Anthropic wrapper (chat, message helpers)
│   ├── chat.py        # base tool-use loop
│   ├── cli_chat.py    # CLI-specific chat (document mentions, commands)
│   ├── cli.py         # prompt-toolkit terminal UI
│   └── tools.py       # ToolManager: aggregates tools across MCP clients
├── pyproject.toml     # deps (uv)
├── uv.lock
└── .env               # config (gitignored — you create/populate this)
```

The starter ships `mcp_server.py` / `mcp_client.py` full of `TODO`s. Each upcoming lesson fills some in: **Defining tools**, **Implementing a client**, **Defining resources**, **Accessing resources**, **Defining prompts**, **Prompts in the client**.

## Adaptations from the course (important for a clean run)

| Course default | This repo | Why |
|----------------|-----------|-----|
| `CLAUDE_MODEL="claude-sonnet-4-5"` | **`claude-haiku-4-5-20251001`** | `core/claude.py` always sends `temperature=1.0`, which **400s on flagship models**; the bare `claude-sonnet-4-5` alias is also risky. Haiku 4.5 supports temperature + tool use and matches the rest of the repo. |
| `USE_UV=1` (uv) | **`USE_UV=0`** (pip) | No `uv` on the dev machine. Pip mode runs the server via `python mcp_server.py`. |
| deps via `uv`/`uv.lock` | shared repo **`.venv`** | `mcp[cli]` + `prompt-toolkit` were added to the repo `requirements.txt` and installed into the shared venv. |

> If you *do* have `uv`, you can ignore all of this: set `USE_UV=1` and run `uv run main.py` — it reads `pyproject.toml`/`uv.lock` and manages its own environment.

## One-time setup for a fresh clone

1. **Install deps** (from the repo root): `pip install -r requirements.txt` into your venv (this pulls `anthropic`, `mcp[cli]`, `prompt-toolkit`, `python-dotenv`, ...).
2. **Create `cli-project/.env`** — it's gitignored, so it won't exist after a clone. It needs:
   ```env
   CLAUDE_MODEL="claude-haiku-4-5-20251001"
   ANTHROPIC_API_KEY="sk-ant-..."
   USE_UV=0
   ```
   ...or just run the **convenience cell** below, which copies your key from the repo-root `.env` and scaffolds the rest.
3. Run the setup check to confirm everything's in place.

## Convenience: scaffold `cli-project/.env` from the repo-root `.env`

If you already have a repo-root `.env` with an `ANTHROPIC_API_KEY` (as the earlier course sections set up), this cell copies that key into `cli-project/.env` and scaffolds `CLAUDE_MODEL` + `USE_UV`. It's **idempotent and safe**: it won't overwrite a key that's already set in the project `.env`, and it never prints the key. Both `.env` files stay gitignored.

In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv

repo_root = Path(os.path.dirname(find_dotenv()))
root_env = repo_root / ".env"
proj_env = repo_root / "building-with-the-claude-api" / "07-mcp" / "cli-project" / ".env"


def read_env(path):
    d = {}
    if path.exists():
        for line in path.read_text().splitlines():
            s = line.strip()
            if s and not s.startswith("#") and "=" in s:
                k, _, v = s.partition("=")
                d[k.strip()] = v.strip().strip('\"')
    return d


proj = read_env(proj_env)
root = read_env(root_env)

# Only fill the key if it's missing/empty in the project .env (never clobber an existing one)
if not proj.get("ANTHROPIC_API_KEY"):
    key = root.get("ANTHROPIC_API_KEY", "")
    if key:
        proj["ANTHROPIC_API_KEY"] = key
        print("Copied ANTHROPIC_API_KEY from repo-root .env")
    else:
        print("No ANTHROPIC_API_KEY in repo-root .env - set it manually in cli-project/.env")
else:
    print("cli-project/.env already has ANTHROPIC_API_KEY - left as-is")

# Scaffold sensible defaults for the project-specific vars if absent
proj.setdefault("CLAUDE_MODEL", "claude-haiku-4-5-20251001")
proj.setdefault("USE_UV", "0")

proj_env.parent.mkdir(parents=True, exist_ok=True)
proj_env.write_text(
    f'CLAUDE_MODEL="{proj["CLAUDE_MODEL"]}"\n'
    f'ANTHROPIC_API_KEY={proj.get("ANTHROPIC_API_KEY", "")}\n'
    "\n"
    "# 1 if using uv, 0 for pip\n"
    f'USE_UV={proj["USE_UV"]}\n'
)

print("Wrote", proj_env)
print("  CLAUDE_MODEL:", proj["CLAUDE_MODEL"])
print("  ANTHROPIC_API_KEY:", "set" if proj.get("ANTHROPIC_API_KEY") else "MISSING")
print("  USE_UV:", proj["USE_UV"])

## Setup check

Verifies the dependencies import and that `cli-project/.env` has the required variables. It reports whether each var is **set** — it never prints the secret value.

In [ ]:
import importlib
import os
from pathlib import Path
from dotenv import find_dotenv

print("Dependencies:")
for pkg in ["anthropic", "mcp", "prompt_toolkit", "dotenv"]:
    try:
        importlib.import_module(pkg)
        print(f"  OK       {pkg}")
    except ImportError:
        print(f"  MISSING  {pkg}  -> pip install -r requirements.txt")

repo_root = Path(os.path.dirname(find_dotenv()))
proj_env = repo_root / "building-with-the-claude-api" / "07-mcp" / "cli-project" / ".env"

print("\nProject .env (", proj_env, "):")
if proj_env.exists():
    present = {}
    for line in proj_env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            present[k.strip()] = bool(v.strip().strip('\"'))
    for need in ["CLAUDE_MODEL", "ANTHROPIC_API_KEY", "USE_UV"]:
        print(f"  {need}: {'set' if present.get(need) else 'MISSING or empty'}")
else:
    print("  NOT FOUND — run the convenience cell above, or create it manually.")

## Running the app

The chatbot is an **interactive terminal app**, so run it in a shell (not in this notebook). In pip mode the server is spawned as `python mcp_server.py`, so the venv must be **activated** first (its `python` must be on PATH):

```powershell
# from the repo root
.\.venv\Scripts\Activate.ps1
cd building-with-the-claude-api\07-mcp\cli-project
python main.py
```

```bash
# uv users (USE_UV=1) can instead just:
uv run main.py
```

At the chat prompt, ask **"what's 1+1?"** — a quick Claude reply confirms the client↔server↔Claude wiring works. Quit with Ctrl-C.

> **Workflow for this section:** each lesson edits `mcp_server.py` / `mcp_client.py` (filling in the `TODO`s). After each change, restart `main.py` and test in the CLI. Right now those files are the **starter** (tools/resources/prompts return empty) — so the bot answers general questions but can't touch documents yet.

Next: **Defining tools with MCP** — implement the read-doc and edit-doc tools on the server.